# Commodity Futures Prices (Corn / Soybeans / Wheat)

**What it is.** Daily futures prices for the grains a Midwest farm markets. **CME Group is the
exchange, but its direct data feeds are paid** (DataMine / real-time API). For *historical
decision-making* the practical free path is **Yahoo Finance continuous front-month contracts**.

| | |
|---|---|
| **Coverage** | CME/CBOT grain & oilseed futures (global benchmark) |
| **Cost / key** | **Free** via Yahoo (`yfinance`) · CME direct = **paid** |
| **Tickers** | Corn `ZC=F` · Soybeans `ZS=F` · Wheat `ZW=F` (¢/bushel) |

**Why it's in Tracera.** Futures are the forward price signal for marketing decisions — when to
sell, hedge, or store. Free daily history is plenty for trend/seasonality analysis.

In [1]:
# Tracera shared setup — credentials + the ONE sample farm location
# (Every source is queried at this same spot so results are comparable.)
import sys, pathlib, requests, pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd()))
from data_sources._common import SAMPLE_FARM, get_key, field_polygon

LAT, LON = SAMPLE_FARM["lat"], SAMPLE_FARM["lon"]
FIPS, STATE, COUNTY = SAMPLE_FARM["fips"], SAMPLE_FARM["state_alpha"], SAMPLE_FARM["county_name"]
print(SAMPLE_FARM["name"], f"| lat={LAT}, lon={LON} | FIPS {FIPS}")

Story County, Iowa (sample farm) | lat=42.05, lon=-93.5 | FIPS 19169


### 1. Connection test — recent corn futures

In [2]:
import yfinance as yf

corn = yf.download("ZC=F", start="2024-01-01", end="2024-02-01", auto_adjust=True, progress=False)
print(f"{len(corn)} trading days. Corn futures are quoted in cents/bushel.")
corn["Close"].tail(3)

21 trading days. Corn futures are quoted in cents/bushel.


Ticker,ZC=F
Date,
2024-01-29,440.25
2024-01-30,447.75
2024-01-31,448.25


### 2. Core query — corn / soy / wheat daily close (multi-year)

In [3]:
TICKERS = {"Corn": "ZC=F", "Soybeans": "ZS=F", "Wheat": "ZW=F"}
px = yf.download(list(TICKERS.values()), start="2021-01-01", end="2024-12-31",
                auto_adjust=True, progress=False)["Close"]
px = px.rename(columns={v: k for k, v in TICKERS.items()})
px_dollars = (px / 100).round(2)   # cents/bu -> $/bu
print("Grain futures front-month close ($/bushel) — recent:")
px_dollars.dropna().tail()

Grain futures front-month close ($/bushel) — recent:


Ticker,Corn,Soybeans,Wheat
Date,,,
2024-12-23,4.48,9.70,5.40
2024-12-24,4.49,9.75,5.35
2024-12-26,4.54,9.88,5.41
2024-12-27,4.54,9.80,5.46
2024-12-30,4.52,9.82,5.48


### Notes & how to extend
- **Units:** corn/soy/wheat quote in **cents/bushel** (÷100 = $/bu). `ZC=F` etc. are Yahoo's
  *continuous front-month* series (auto-rolled), fine for trend/seasonality, not for
  contract-precise basis work.
- **Other contracts:** soybean oil `ZL=F`, meal `ZM=F`, oats `ZO=F`, rough rice `ZR=F`,
  live cattle `LE=F`, lean hogs `HE=F`.
- **Basis:** local cash price (from **AMS Market News**) minus the futures price = basis, the
  number a farmer actually negotiates.
- **Paid upgrades if needed:** **Databento** (clean historical futures, pay-as-you-go),
  **Barchart**, or **CME DataMine** for settlement-accurate, contract-level data.